In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.2 MB/s eta 0:00:00


In [2]:
import os
import copy
import time
import math
import random
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from PIL import Image
from ultralytics import YOLO

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("torch:", torch.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch: 2.11.0+cu128


In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [6]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"
POSITION_ROOT = "/content/drive/MyDrive/scenario23_paper_style_sequence_split"
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"

FULL_CSV_PATH = os.path.join(AUTHOR_ROOT, "scenario23_img_beam.csv")

EXTRACT_ROOT = "/content/scenario23_dev_w_resources"
YOLO_CROP_ROOT = "/content/yolo_uav_crops_scenario23"

INDEX_COL = "index"
IMAGE_COL = "unit1_rgb"
LABEL_COL = "unit1_beam"

print("AUTHOR_ROOT:", os.path.exists(AUTHOR_ROOT), AUTHOR_ROOT)
print("POSITION_ROOT:", os.path.exists(POSITION_ROOT), POSITION_ROOT)
print("ZIP_PATH:", os.path.exists(ZIP_PATH), ZIP_PATH)
print("FULL_CSV_PATH:", os.path.exists(FULL_CSV_PATH), FULL_CSV_PATH)

assert os.path.exists(AUTHOR_ROOT)
assert os.path.exists(POSITION_ROOT)
assert os.path.exists(ZIP_PATH)
assert os.path.exists(FULL_CSV_PATH)

AUTHOR_ROOT: True /content/drive/MyDrive/Image beam
POSITION_ROOT: True /content/drive/MyDrive/scenario23_paper_style_sequence_split
ZIP_PATH: True /content/drive/MyDrive/scenario23_dev_w_resources.zip
FULL_CSV_PATH: True /content/drive/MyDrive/Image beam/scenario23_img_beam.csv


In [7]:
X_pos_train = np.load(os.path.join(POSITION_ROOT, "X_train_seq.npy"))
y_train = np.load(os.path.join(POSITION_ROOT, "y_train_seq.npy"))

X_pos_val = np.load(os.path.join(POSITION_ROOT, "X_val_seq.npy"))
y_val = np.load(os.path.join(POSITION_ROOT, "y_val_seq.npy"))

X_pos_test = np.load(os.path.join(POSITION_ROOT, "X_test_seq.npy"))
y_test = np.load(os.path.join(POSITION_ROOT, "y_test_seq.npy"))

print("Position sequences:")
print("train:", X_pos_train.shape, y_train.shape)
print("val  :", X_pos_val.shape, y_val.shape)
print("test :", X_pos_test.shape, y_test.shape)

assert X_pos_train.shape == (7968, 4, 21)
assert X_pos_val.shape == (2276, 4, 21)
assert X_pos_test.shape == (1139, 4, 21)

num_classes = int(max(y_train.max(), y_val.max(), y_test.max()) + 1)
pos_input_dim = X_pos_train.shape[-1]

print("num_classes:", num_classes)
print("pos_input_dim:", pos_input_dim)

Position sequences:
train: (7968, 4, 21) (7968,)
val  : (2276, 4, 21) (2276,)
test : (1139, 4, 21) (1139,)
num_classes: 29
pos_input_dim: 21


In [8]:
full_img_df = pd.read_csv(FULL_CSV_PATH)
full_img_df = full_img_df.sort_values(INDEX_COL).reset_index(drop=True)

print("Full image CSV:", full_img_df.shape)
print("Columns:", full_img_df.columns.tolist())

for col in [INDEX_COL, IMAGE_COL, LABEL_COL]:
    assert col in full_img_df.columns, f"Missing column: {col}"

display(full_img_df.head())
display(full_img_df.tail())

Full image CSV: (11387, 3)
Columns: ['index', 'unit1_rgb', 'unit1_beam']


,index,unit1_rgb,unit1_beam
0,1,../scenario23_dev/unit1/camera_data/image_BS1_...,22
1,2,../scenario23_dev/unit1/camera_data/image_BS1_...,22
2,3,../scenario23_dev/unit1/camera_data/image_BS1_...,22
3,4,../scenario23_dev/unit1/camera_data/image_BS1_...,22
4,5,../scenario23_dev/unit1/camera_data/image_BS1_...,20


,index,unit1_rgb,unit1_beam
11382,11383,../scenario23_dev/unit1/camera_data/image_BS1_...,20
11383,11384,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11384,11385,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11385,11386,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11386,11387,../scenario23_dev/unit1/camera_data/image_BS1_...,19


In [9]:
os.makedirs(EXTRACT_ROOT, exist_ok=True)

marker_file = os.path.join(EXTRACT_ROOT, ".extracted_done")

if not os.path.exists(marker_file):
    print("Extracting scenario23 zip. Wait...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_ROOT)

    with open(marker_file, "w") as f:
        f.write("done")

    print("Extraction completed.")
else:
    print("Already extracted. Skipping.")

print("Extract root sample:")
print(os.listdir(EXTRACT_ROOT)[:30])

Extracting scenario23 zip. Wait...
Extraction completed.
Extract root sample:
['.extracted_done', 'scenario23_dev']


In [10]:
image_files = []

for root, dirs, files in os.walk(EXTRACT_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            image_files.append(os.path.join(root, f))

print("Total extracted image files:", len(image_files))
for p in image_files[:10]:
    print(p)

assert len(image_files) > 0, "No images found after extraction."

image_lookup_basename = {}
image_lookup_suffix = {}

for p in image_files:
    norm_p = p.replace("\\", "/")
    base = os.path.basename(norm_p)
    image_lookup_basename[base] = p

    parts = norm_p.split("/")
    for n in [2, 3, 4, 5, 6, 7, 8]:
        if len(parts) >= n:
            image_lookup_suffix["/".join(parts[-n:])] = p

def resolve_raw_image_path(path_from_csv):
    path_from_csv = str(path_from_csv).replace("\\", "/")
    clean = path_from_csv.lstrip("./").lstrip("/")

    if os.path.exists(path_from_csv):
        return path_from_csv

    candidate = os.path.join(EXTRACT_ROOT, clean)
    if os.path.exists(candidate):
        return candidate

    base = os.path.basename(clean)
    if base in image_lookup_basename:
        return image_lookup_basename[base]

    parts = clean.split("/")
    for n in [8, 7, 6, 5, 4, 3, 2]:
        if len(parts) >= n:
            suffix = "/".join(parts[-n:])
            if suffix in image_lookup_suffix:
                return image_lookup_suffix[suffix]

    return None

full_img_df["raw_image_path"] = full_img_df[IMAGE_COL].apply(resolve_raw_image_path)

missing = full_img_df["raw_image_path"].isna().sum()
print("Missing raw image paths:", missing, "of", len(full_img_df))

assert missing == 0

display(full_img_df[[INDEX_COL, IMAGE_COL, "raw_image_path", LABEL_COL]].head())

Total extracted image files: 11387
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_1131_17_01_30.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_11340_18_00_18.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_126_16_58_58.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_3501_17_08_06.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_9950_17_55_54.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_6504_17_45_27.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_631_17_00_08.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_1627_17_02_52.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_5079_17_13_56.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_2855_17_06_17.

,index,unit1_rgb,raw_image_path,unit1_beam
0,1,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
1,2,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
2,3,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
3,4,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
4,5,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,20


In [ ]:
yolo_model = YOLO("yolov8n.pt")

os.makedirs(YOLO_CROP_ROOT, exist_ok=True)

def yolo_crop_image(image_path, save_root=YOLO_CROP_ROOT, conf=0.10, padding=30):
    image_path = str(image_path)

    base = os.path.basename(image_path)
    save_path = os.path.join(save_root, base)

    if os.path.exists(save_path):
        return save_path

    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    results = yolo_model.predict(source=image_path, conf=conf, verbose=False)
    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        img.save(save_path)
        return save_path

    xyxy = boxes.xyxy.cpu().numpy()
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    best_idx = areas.argmax()

    x1, y1, x2, y2 = xyxy[best_idx]

    x1 = max(int(x1) - padding, 0)
    y1 = max(int(y1) - padding, 0)
    x2 = min(int(x2) + padding, w)
    y2 = min(int(y2) + padding, h)

    crop = img.crop((x1, y1, x2, y2))

    if crop.size[0] < 10 or crop.size[1] < 10:
        img.save(save_path)
        return save_path

    crop.save(save_path)
    return save_path

sample_raw = full_img_df["raw_image_path"].iloc[0]
sample_crop = yolo_crop_image(sample_raw)

print("Sample raw:", sample_raw)
print("Sample crop:", sample_crop)

plt.figure(figsize=(5, 5))
plt.imshow(Image.open(sample_raw).convert("RGB"))
plt.axis("off")
plt.title("Raw Image")
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(Image.open(sample_crop).convert("RGB"))
plt.axis("off")
plt.title("YOLO Crop / Fallback")
plt.show()

In [11]:
crop_paths = []

start = time.time()

for i, p in enumerate(full_img_df["raw_image_path"]):
    crop_paths.append(yolo_crop_image(p))

    if (i + 1) % 500 == 0:
        elapsed = (time.time() - start) / 60
        print(i + 1, "/", len(full_img_df), "| elapsed min:", round(elapsed, 2))

full_img_df["image_path"] = crop_paths

print("YOLO crop/fallback completed:", len(crop_paths))

500 / 11387 | elapsed min: 0.0
1000 / 11387 | elapsed min: 0.0
1500 / 11387 | elapsed min: 0.0
2000 / 11387 | elapsed min: 0.0
2500 / 11387 | elapsed min: 0.0
3000 / 11387 | elapsed min: 0.0
3500 / 11387 | elapsed min: 0.0
4000 / 11387 | elapsed min: 0.0
4500 / 11387 | elapsed min: 0.0
5000 / 11387 | elapsed min: 0.0
5500 / 11387 | elapsed min: 0.0
6000 / 11387 | elapsed min: 0.0
6500 / 11387 | elapsed min: 0.0
7000 / 11387 | elapsed min: 0.0
7500 / 11387 | elapsed min: 0.0
8000 / 11387 | elapsed min: 0.0
8500 / 11387 | elapsed min: 0.0
9000 / 11387 | elapsed min: 0.0
9500 / 11387 | elapsed min: 0.0
10000 / 11387 | elapsed min: 0.0
10500 / 11387 | elapsed min: 0.0
11000 / 11387 | elapsed min: 0.0
YOLO crop/fallback completed: 11387


In [12]:
crop_paths = []

start = time.time()

for i, p in enumerate(full_img_df["raw_image_path"]):
    crop_paths.append(yolo_crop_image(p))

    if (i + 1) % 500 == 0:
        elapsed = (time.time() - start) / 60
        print(i + 1, "/", len(full_img_df), "| elapsed min:", round(elapsed, 2))

full_img_df["image_path"] = crop_paths

print("YOLO crop/fallback completed:", len(crop_paths))

500 / 11387 | elapsed min: 0.0
1000 / 11387 | elapsed min: 0.0
1500 / 11387 | elapsed min: 0.0
2000 / 11387 | elapsed min: 0.0
2500 / 11387 | elapsed min: 0.0
3000 / 11387 | elapsed min: 0.0
3500 / 11387 | elapsed min: 0.0
4000 / 11387 | elapsed min: 0.0
4500 / 11387 | elapsed min: 0.0
5000 / 11387 | elapsed min: 0.0
5500 / 11387 | elapsed min: 0.0
6000 / 11387 | elapsed min: 0.0
6500 / 11387 | elapsed min: 0.0
7000 / 11387 | elapsed min: 0.0
7500 / 11387 | elapsed min: 0.0
8000 / 11387 | elapsed min: 0.0
8500 / 11387 | elapsed min: 0.0
9000 / 11387 | elapsed min: 0.0
9500 / 11387 | elapsed min: 0.0
10000 / 11387 | elapsed min: 0.0
10500 / 11387 | elapsed min: 0.0
11000 / 11387 | elapsed min: 0.0
YOLO crop/fallback completed: 11387


In [13]:
def build_image_sequences_from_full_df(df, seq_len=4):
    df = df.sort_values(INDEX_COL).reset_index(drop=True).copy()

    idx_values = df[INDEX_COL].astype(int).values
    img_paths = df["image_path"].astype(str).values

    X_img_seq = []
    target_indices = []

    for i in range(seq_len - 1, len(df) - 1):
        past_idx = idx_values[i - seq_len + 1 : i + 1]
        target_idx = idx_values[i + 1]

        all_idx = np.concatenate([past_idx, [target_idx]])
        expected_idx = np.arange(all_idx[0], all_idx[0] + seq_len + 1)

        if np.array_equal(all_idx, expected_idx):
            X_img_seq.append(img_paths[i - seq_len + 1 : i + 1])
            target_indices.append(target_idx)

    return np.array(X_img_seq, dtype=object), np.array(target_indices, dtype=np.int64)

SEQ_LEN = 4

X_img_all, target_idx_all = build_image_sequences_from_full_df(full_img_df, seq_len=SEQ_LEN)

print("X_img_all:", X_img_all.shape)
print("target_idx_all:", target_idx_all.shape)

total_pos = len(X_pos_train) + len(X_pos_val) + len(X_pos_test)
print("Total position sequences:", total_pos)

assert len(X_img_all) >= total_pos

X_img_all: (11383, 4)
target_idx_all: (11383,)
Total position sequences: 11383


In [14]:
# Build labels from full CSV using same t+1 target index as X_img_all

full_img_df = full_img_df.sort_values(INDEX_COL).reset_index(drop=True)

idx_values = full_img_df[INDEX_COL].astype(int).values
labels_raw = full_img_df[LABEL_COL].astype(int).values

# Remap original beam labels to 0..C-1
all_beam_labels = sorted(full_img_df[LABEL_COL].astype(int).unique())
label_to_id = {label: i for i, label in enumerate(all_beam_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

y_all = []
target_idx_label = []

SEQ_LEN = 4

for i in range(SEQ_LEN - 1, len(full_img_df) - 1):
    past_idx = idx_values[i - SEQ_LEN + 1 : i + 1]
    target_idx = idx_values[i + 1]

    all_idx = np.concatenate([past_idx, [target_idx]])
    expected_idx = np.arange(all_idx[0], all_idx[0] + SEQ_LEN + 1)

    if np.array_equal(all_idx, expected_idx):
        original_label = int(labels_raw[i + 1])
        y_all.append(label_to_id[original_label])
        target_idx_label.append(target_idx)

y_all = np.array(y_all, dtype=np.int64)
target_idx_label = np.array(target_idx_label, dtype=np.int64)

print("X_img_all:", X_img_all.shape)
print("y_all:", y_all.shape)
print("target_idx_label:", target_idx_label.shape)
print("num_classes:", len(all_beam_labels))

assert len(X_img_all) == len(y_all)

X_img_all: (11383, 4)
y_all: (11383,)
target_idx_label: (11383,)
num_classes: 29


In [15]:
# Recombine position NPY into one full sequence pool
# This preserves all 11383 position sequences but ignores old train/val/test order problem.

X_pos_all = np.concatenate([X_pos_train, X_pos_val, X_pos_test], axis=0)

print("X_pos_all:", X_pos_all.shape)
print("y_all:", y_all.shape)
print("X_img_all:", X_img_all.shape)

assert len(X_pos_all) == len(X_img_all) == len(y_all)

X_pos_all: (11383, 4, 21)
y_all: (11383,)
X_img_all: (11383, 4)


In [16]:
from sklearn.model_selection import train_test_split

all_indices = np.arange(len(y_all))

idx_train, idx_temp, y_train_tmp, y_temp = train_test_split(
    all_indices,
    y_all,
    test_size=0.30,
    random_state=SEED,
    stratify=y_all
)

idx_val, idx_test, y_val_tmp, y_test_tmp = train_test_split(
    idx_temp,
    y_temp,
    test_size=1/3,
    random_state=SEED,
    stratify=y_temp
)

X_img_train = X_img_all[idx_train]
X_pos_train = X_pos_all[idx_train]
y_train = y_all[idx_train]

X_img_val = X_img_all[idx_val]
X_pos_val = X_pos_all[idx_val]
y_val = y_all[idx_val]

X_img_test = X_img_all[idx_test]
X_pos_test = X_pos_all[idx_test]
y_test = y_all[idx_test]

num_classes = len(all_beam_labels)
pos_input_dim = X_pos_train.shape[-1]

print("New jointly aligned split:")
print("train:", X_img_train.shape, X_pos_train.shape, y_train.shape)
print("val  :", X_img_val.shape, X_pos_val.shape, y_val.shape)
print("test :", X_img_test.shape, X_pos_test.shape, y_test.shape)

print("num_classes:", num_classes)

New jointly aligned split:
train: (7968, 4) (7968, 4, 21) (7968,)
val  : (2276, 4) (2276, 4, 21) (2276,)
test : (1139, 4) (1139, 4, 21) (1139,)
num_classes: 29


eta dorkar nai run dewar 

In [17]:
n_train = len(X_pos_train)
n_val = len(X_pos_val)
n_test = len(X_pos_test)

total_needed = n_train + n_val + n_test

print("Image sequences available:", len(X_img_all))
print("Position total needed:", total_needed)

X_img_train = X_img_all[:n_train]
X_img_val = X_img_all[n_train:n_train+n_val]
X_img_test = X_img_all[n_train+n_val:n_train+n_val+n_test]

print("Final aligned shapes:")
print("train:", X_img_train.shape, X_pos_train.shape, y_train.shape)
print("val  :", X_img_val.shape, X_pos_val.shape, y_val.shape)
print("test :", X_img_test.shape, X_pos_test.shape, y_test.shape)

assert len(X_img_train) == len(X_pos_train) == len(y_train)
assert len(X_img_val) == len(X_pos_val) == len(y_val)
assert len(X_img_test) == len(X_pos_test) == len(y_test)

Image sequences available: 11383
Position total needed: 11383
Final aligned shapes:
train: (7968, 4) (7968, 4, 21) (7968,)
val  : (2276, 4) (2276, 4, 21) (2276,)
test : (1139, 4) (1139, 4, 21) (1139,)


In [18]:
train_img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10)
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class PaperReplicaDataset(Dataset):
    def __init__(self, img_seq_paths, pos_seq, labels, transform=None):
        self.img_seq_paths = img_seq_paths
        self.pos_seq = torch.tensor(pos_seq, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def load_image(self, path):
        img = Image.open(path).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        return img

    def __getitem__(self, idx):
        imgs = [self.load_image(p) for p in self.img_seq_paths[idx]]
        imgs = torch.stack(imgs, dim=0)
        pos = self.pos_seq[idx]
        label = self.labels[idx]
        return imgs, pos, label

BATCH_SIZE = 8

train_dataset = PaperReplicaDataset(X_img_train, X_pos_train, y_train, transform=train_img_transform)
val_dataset = PaperReplicaDataset(X_img_val, X_pos_val, y_val, transform=eval_img_transform)
test_dataset = PaperReplicaDataset(X_img_test, X_pos_test, y_test, transform=eval_img_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

imgs_b, pos_b, y_b = next(iter(train_loader))

print("imgs_b:", imgs_b.shape)
print("pos_b:", pos_b.shape)
print("y_b:", y_b.shape)

imgs_b: torch.Size([8, 4, 3, 224, 224])
pos_b: torch.Size([8, 4, 21])
y_b: torch.Size([8])


In [19]:
class PositionOnlyLSTM(nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128, dropout=0.25):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(128, hidden_dim, batch_first=True)

        self.cls = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, pos):
        x = self.proj(pos)
        out, _ = self.lstm(x)
        return self.cls(out[:, -1, :])

class PosDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

pos_train_loader = DataLoader(PosDataset(X_pos_train, y_train), batch_size=128, shuffle=True, num_workers=0)
pos_val_loader = DataLoader(PosDataset(X_pos_val, y_val), batch_size=128, shuffle=False, num_workers=0)

def eval_pos_model(model, loader):
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            pred = logits.argmax(dim=1)

            total += y.size(0)
            correct += (pred == y).sum().item()

    return 100.0 * correct / total

pos_model = PositionOnlyLSTM(
    input_dim=pos_input_dim,
    num_classes=num_classes
).to(device)

opt = optim.AdamW(pos_model.parameters(), lr=1e-3, weight_decay=1e-3)
crit = nn.CrossEntropyLoss(label_smoothing=0.05)

for epoch in range(1, 16):
    pos_model.train()

    for x, y in pos_train_loader:
        x = x.to(device)
        y = y.to(device)

        opt.zero_grad()
        loss = crit(pos_model(x), y)
        loss.backward()
        opt.step()

    val_acc = eval_pos_model(pos_model, pos_val_loader)
    print(f"Position-only epoch {epoch:02d} | val acc: {val_acc:.2f}%")

Position-only epoch 01 | val acc: 53.56%
Position-only epoch 02 | val acc: 56.37%
Position-only epoch 03 | val acc: 60.41%
Position-only epoch 04 | val acc: 60.11%
Position-only epoch 05 | val acc: 62.13%
Position-only epoch 06 | val acc: 62.74%
Position-only epoch 07 | val acc: 62.87%
Position-only epoch 08 | val acc: 63.97%
Position-only epoch 09 | val acc: 63.66%
Position-only epoch 10 | val acc: 62.30%
Position-only epoch 11 | val acc: 65.20%
Position-only epoch 12 | val acc: 65.47%
Position-only epoch 13 | val acc: 64.41%
Position-only epoch 14 | val acc: 66.04%
Position-only epoch 15 | val acc: 65.64%


In [20]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced = max(channels // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(channels, reduced, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y

class SEBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1, downsample=None, reduction=16):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.se = SEBlock(out_channels, reduction)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.relu(out + identity)
        return out

class SEResNet34Encoder(nn.Module):
    def __init__(self, output_dim=128, dropout=0.35):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)

        self.layer1 = self._make_layer(64, 3, stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.proj = nn.Sequential(
            nn.Linear(512, output_dim),
            nn.LayerNorm(output_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self._init_weights()

    def _make_layer(self, out_channels, blocks, stride):
        downsample = None

        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        layers = [SEBasicBlock(self.in_channels, out_channels, stride, downsample)]
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(SEBasicBlock(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return self.proj(x)

In [21]:
class CAFormerBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=4, dropout=0.25):
        super().__init__()

        self.img_query_pos_kv = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.pos_query_img_kv = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.img_norm1 = nn.LayerNorm(embed_dim)
        self.pos_norm1 = nn.LayerNorm(embed_dim)

        self.img_ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim)
        )

        self.pos_ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim)
        )

        self.img_norm2 = nn.LayerNorm(embed_dim)
        self.pos_norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, img_tokens, pos_tokens):
        img_cross, _ = self.img_query_pos_kv(
            query=img_tokens,
            key=pos_tokens,
            value=pos_tokens
        )

        pos_cross, _ = self.pos_query_img_kv(
            query=pos_tokens,
            key=img_tokens,
            value=img_tokens
        )

        img_tokens = self.img_norm1(img_tokens + self.dropout(img_cross))
        pos_tokens = self.pos_norm1(pos_tokens + self.dropout(pos_cross))

        img_tokens = self.img_norm2(img_tokens + self.dropout(self.img_ffn(img_tokens)))
        pos_tokens = self.pos_norm2(pos_tokens + self.dropout(self.pos_ffn(pos_tokens)))

        return img_tokens, pos_tokens

In [22]:
class PaperReplicaBeamTrackerV2(nn.Module):
    def __init__(
        self,
        pos_input_dim,
        num_classes,
        embed_dim=128,
        num_heads=4,
        ca_layers=1,
        lstm_hidden_dim=128,
        dropout=0.35
    ):
        super().__init__()

        self.image_encoder = SEResNet34Encoder(
            output_dim=embed_dim,
            dropout=dropout
        )

        self.position_encoder = nn.Sequential(
            nn.Linear(pos_input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU()
        )

        self.caformer_blocks = nn.ModuleList([
            CAFormerBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
            for _ in range(ca_layers)
        ])

        self.image_gate = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.Sigmoid()
        )

        self.temporal_lstm = nn.LSTM(
            input_size=embed_dim * 2,
            hidden_size=lstm_hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, img_seq, pos_seq):
        B, T, C, H, W = img_seq.shape

        img_flat = img_seq.reshape(B * T, C, H, W)

        img_features = self.image_encoder(img_flat)
        img_tokens = img_features.reshape(B, T, -1)

        pos_tokens = self.position_encoder(pos_seq)

        for block in self.caformer_blocks:
            img_tokens, pos_tokens = block(img_tokens, pos_tokens)

        gate = self.image_gate(pos_tokens)
        img_tokens = img_tokens * gate

        fused_tokens = torch.cat([img_tokens, pos_tokens], dim=-1)

        temporal_out, _ = self.temporal_lstm(fused_tokens)

        logits = self.classifier(temporal_out[:, -1, :])

        return logits

In [23]:
paper_model = PaperReplicaBeamTrackerV2(
    pos_input_dim=pos_input_dim,
    num_classes=num_classes,
    embed_dim=128,
    num_heads=4,
    ca_layers=1,
    lstm_hidden_dim=128,
    dropout=0.35
).to(device)

imgs_tmp = imgs_b.to(device)
pos_tmp = pos_b.to(device)

with torch.no_grad():
    logits_tmp = paper_model(imgs_tmp, pos_tmp)

print("logits:", logits_tmp.shape)
print("expected:", torch.Size([imgs_tmp.size(0), num_classes]))

logits: torch.Size([8, 29])
expected: torch.Size([8, 29])


In [24]:
def calculate_top1_dba_from_prediction(logits, labels, num_classes):
    pred_ids = torch.argmax(logits, dim=1)

    dba_scores = []

    for pred_id, true_id in zip(pred_ids, labels):
        pred_id = int(pred_id.item())
        true_id = int(true_id.item())

        diff = abs(pred_id - true_id)
        circular_diff = min(diff, num_classes - diff)

        score = 1.0 - (circular_diff / (num_classes / 2.0))
        score = max(score, 0.0)

        dba_scores.append(score)

    return sum(dba_scores) / len(dba_scores)

def evaluate_model(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    total = 0
    loss_sum = 0.0
    correct = {k: 0 for k in ks}
    dba_sum = 0.0

    with torch.no_grad():
        for imgs, pos, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            pos = pos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(imgs, pos)
            loss = criterion(logits, labels)

            bs = labels.size(0)
            total += bs
            loss_sum += loss.item() * bs

            max_k = max(ks)
            _, pred = torch.topk(logits, k=max_k, dim=1)
            pred_t = pred.t()

            for k in ks:
                correct[k] += pred_t[:k].eq(labels.view(1, -1)).sum().item()

            batch_dba = calculate_top1_dba_from_prediction(
                logits=logits,
                labels=labels,
                num_classes=logits.shape[1]
            )

            dba_sum += batch_dba * bs

    metrics = {"loss": loss_sum / total}

    for k in ks:
        metrics[f"top{k}"] = 100.0 * correct[k] / total

    metrics["dba"] = dba_sum / total

    return metrics

In [ ]:
def train_paper_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=20,
    lr=3e-4,
    weight_decay=2e-3,
    label_smoothing=0.03,
    patience=6,
    save_path="/content/drive/MyDrive/best_paper_replica_v2.pth"
):
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    best_val_loss = float("inf")
    patience_counter = 0

    history = []

    for epoch in range(1, epochs + 1):
        model.train()

        train_total = 0
        train_loss_sum = 0.0
        train_correct = 0

        for imgs, pos, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            pos = pos.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            logits = model(imgs, pos)
            loss = criterion(logits, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            bs = labels.size(0)
            train_total += bs
            train_loss_sum += loss.item() * bs
            train_correct += (logits.argmax(dim=1) == labels).sum().item()

        train_loss = train_loss_sum / train_total
        train_top1 = 100.0 * train_correct / train_total

        val_metrics = evaluate_model(model, val_loader, device, ks=(1, 2, 3, 5))

        val_loss = val_metrics["loss"]
        scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_top1": train_top1,
            "val_loss": val_loss,
            "val_top1": val_metrics["top1"],
            "val_top2": val_metrics["top2"],
            "val_top3": val_metrics["top3"],
            "val_top5": val_metrics["top5"],
            "val_dba": val_metrics["dba"],
        })

        print(
            f"Epoch {epoch:03d} | LR {current_lr:.2e} | "
            f"Train Loss {train_loss:.4f} | Train Top1 {train_top1:.2f} | "
            f"Val Loss {val_loss:.4f} | Val Top1 {val_metrics['top1']:.2f} | "
            f"Top2 {val_metrics['top2']:.2f} | "
            f"Top3 {val_metrics['top3']:.2f} | "
            f"Top5 {val_metrics['top5']:.2f} | "
            f"DBA {val_metrics['dba']:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping")
            break

    return pd.DataFrame(history)

In [ ]:
set_seed(SEED)

paper_model = PaperReplicaBeamTrackerV2(
    pos_input_dim=pos_input_dim,
    num_classes=num_classes,
    embed_dim=128,
    num_heads=4,
    ca_layers=1,
    lstm_hidden_dim=128,
    dropout=0.35
).to(device)

paper_model_path = "/content/drive/MyDrive/best_full_paper_replica_v2_yolo_se_caformer_lstm.pth"

history = train_paper_model(
    model=paper_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=20,
    lr=3e-4,
    weight_decay=2e-3,
    label_smoothing=0.03,
    patience=20,
    save_path=paper_model_path
)

Epoch 001 | LR 3.00e-04 | Train Loss 1.7675 | Train Top1 46.77 | Val Loss 1.2653 | Val Top1 55.27 | Top2 79.26 | Top3 86.86 | Top5 94.33 | DBA 0.9420
Saved best model
Epoch 002 | LR 3.00e-04 | Train Loss 1.4450 | Train Top1 53.51 | Val Loss 1.2025 | Val Top1 57.21 | Top2 79.75 | Top3 88.58 | Top5 95.08 | DBA 0.9479
Saved best model
Epoch 003 | LR 3.00e-04 | Train Loss 1.3798 | Train Top1 55.48 | Val Loss 1.1902 | Val Top1 57.78 | Top2 80.18 | Top3 89.02 | Top5 95.21 | DBA 0.9463
Saved best model
Epoch 004 | LR 3.00e-04 | Train Loss 1.3416 | Train Top1 57.08 | Val Loss 1.1238 | Val Top1 58.92 | Top2 81.59 | Top3 90.60 | Top5 96.35 | DBA 0.9504
Saved best model
Epoch 005 | LR 3.00e-04 | Train Loss 1.3167 | Train Top1 57.47 | Val Loss 1.1382 | Val Top1 59.45 | Top2 82.16 | Top3 90.16 | Top5 96.05 | DBA 0.9517
Patience: 1/6
Epoch 006 | LR 3.00e-04 | Train Loss 1.2781 | Train Top1 59.35 | Val Loss 1.0509 | Val Top1 62.39 | Top2 83.74 | Top3 91.48 | Top5 96.84 | DBA 0.9549
Saved best model
E

In [ ]:
paper_model_eval = PaperReplicaBeamTrackerV2(
    pos_input_dim=pos_input_dim,
    num_classes=num_classes,
    embed_dim=128,
    num_heads=4,
    ca_layers=1,
    lstm_hidden_dim=128,
    dropout=0.35
).to(device)

paper_model_eval.load_state_dict(torch.load(paper_model_path, map_location=device))

test_metrics = evaluate_model(
    paper_model_eval,
    test_loader,
    device,
    ks=(1, 2, 3, 5)
)

print("Final Test Metrics:")
print(test_metrics)

print(f"Top-1: {test_metrics['top1']:.2f}%")
print(f"Top-2: {test_metrics['top2']:.2f}%")
print(f"Top-3: {test_metrics['top3']:.2f}%")
print(f"Top-5: {test_metrics['top5']:.2f}%")
print(f"DBA  : {test_metrics['dba']:.4f}")
print(f"Loss : {test_metrics['loss']:.4f}")

In [ ]:
result_df = pd.DataFrame([
    {
        "Model": "YOLOv8 + SE-ResNet34 + CAFormer + LSTM V2",
        "Top-1": test_metrics["top1"],
        "Top-2": test_metrics["top2"],
        "Top-3": test_metrics["top3"],
        "Top-5": test_metrics["top5"],
        "DBA": test_metrics["dba"],
        "Loss": test_metrics["loss"],
    }
])

display(result_df)

RESULT_PATH = "/content/drive/MyDrive/full_paper_replica_v2_result_with_dba.csv"
result_df.to_csv(RESULT_PATH, index=False)

print("Saved:", RESULT_PATH)